# 03 — Pilot đo tỷ lệ nhãn và chia tập dữ liệu

Trước khi chia Train / Validation / Hold-out, tôi cần biết mỗi nhãn chiếm bao nhiêu phần trong dữ liệu. Kích thước Hold-out
được tính theo lớp hiếm nhất (N ≈ m / p_rare với m = 80, trần 1.000 — Q8), nên nếu đoán sai tỷ lệ thì Hold-out có thể có quá ít
review FEATURE hoặc UX để đánh giá đáng tin. Calibration cho tôi một ước lượng thô (trên 150 review ngẫu nhiên chỉ có khoảng 5%
BUG, 5% FEATURE, 7% UX), nhưng con số đó quá nhỏ để dựa vào. Vì vậy bước đầu tiên của notebook này là một pilot 400 review.

Một vài điểm về thiết kế:

- Pilot được rút **ngẫu nhiên thuần** (seed 42) từ corpus Shopee trung gian, sau khi loại 180 review calibration. Tôi không chọn
  theo từ khóa, không bỏ review trùng nội dung, không lọc theo số sao, để tỷ lệ đo được phản ánh đúng dữ liệu (corpus gần như toàn
  bộ review 12 tháng — amendment v1.5).
- Pilot là một phần của Hold-out (Q9). Vì vậy các file pilot thuộc diện khóa của `HOLDOUT_LOCK.md`: hai người gán độc lập bằng
  guideline v1.2 đã đóng băng, ca lệch cũng chốt theo đúng guideline đó, và tôi không dùng pilot để sửa guideline hay làm ví dụ.
- Bảng gán chỉ có `review_id` và nội dung review; số sao, ngày, phiên bản app được ẩn (Q17).

Ở nhịp này tôi mới rút mẫu và chuẩn bị phần đo. Việc tính kích thước Hold-out và chia tập sẽ làm sau, khi đã có tỷ lệ từ pilot.
Phần xử lý nằm trong `src/sampling/pilot.py`.

In [1]:
import sys, os, json
sys.path.append(os.path.abspath(os.path.join('..', '02_src')))
import pandas as pd
from src.utils.kaggle_paths import input_dir
from src.utils.seeds import set_seed
from src.preprocessing.clean import load_jsonl
from src.annotation.calibration import export_blank
from src.annotation.agreement import merge, agreement, disagreements_by_pair, build_finalize_rows, write_finalize
from src.sampling.pilot import (draw_pilot, write_pilot_manifest, prevalence_table, prevalence_bounds, rarest,
                                INSTRUCTIONS)
set_seed(42)
TAG = 'T0_2026-10-06'
SOURCE = os.path.join(input_dir('interim'), f'shopee_clean_{TAG}.jsonl')
EXCLUSION = os.path.join(input_dir('manifests'), f'holdout_exclusion_{TAG}.json')
PILOT_MANIFEST = os.path.join(input_dir('manifests'), f'pilot_manifest_{TAG}.json')
PILOT_DIR = os.path.abspath(os.path.join('..', '07_human_eval', 'pilot'))
PILOT_SHEET = os.path.join(PILOT_DIR, 'pilot_blank.csv')

## Rút mẫu pilot

In [2]:
records = load_jsonl(SOURCE)
excluded = json.load(open(EXCLUSION, encoding='utf-8'))['review_ids']
if os.path.exists(PILOT_SHEET) and os.path.exists(PILOT_MANIFEST):
    print('Pilot đã được rút trước đó — giữ nguyên (không ghi đè).')
else:
    pilot_rows, pool_size = draw_pilot(records, n=400, exclude_ids=excluded, seed=42)
    export_blank(pilot_rows, PILOT_SHEET)
    write_pilot_manifest(pilot_rows, pool_size, SOURCE, EXCLUSION, PILOT_SHEET, PILOT_MANIFEST, seed=42)
pilot_manifest = json.load(open(PILOT_MANIFEST, encoding='utf-8'))
pilot_ids = pilot_manifest['review_ids']
print(f"Pool sau khi loại calibration: {pilot_manifest['pool_size']:,} review (corpus {len(records):,} − {len(excluded)})")
print(f'Pilot: {len(pilot_ids)} review → {PILOT_SHEET}')
print('Trùng với calibration:', len(set(pilot_ids) & set(excluded)), '| review_id trùng trong pilot:', len(pilot_ids) - len(set(pilot_ids)))
print()
print(INSTRUCTIONS)

Pool sau khi loại calibration: 55,870 review (corpus 56,050 − 180)
Pilot: 400 review → C:\Workspace\kltn-vn-app-review-requirements\07_human_eval\pilot\pilot_blank.csv
Trùng với calibration: 0 | review_id trùng trong pilot: 0

Pilot prevalence — 400 review ngẫu nhiên, gán bằng guideline v1.2 (FROZEN)
- Nhãn: BUG / FEATURE / UX / OTHER; cột ambiguous: 1 khi ≥2 ý actionable khác lớp không ý nào trội, ngược lại 0; note: lý do khi phân vân.
- Hai người gán độc lập trên hai bản sao Google Sheets riêng (pilot_A / pilot_B), không xem bản của nhau.
- Pilot là một phần của Hold-out: không thảo luận nội dung pilot để sửa guideline; ca lệch được chốt theo guideline v1.2.
- Gán xong tải về pilot_A.csv và pilot_B.csv, đặt trong 07_human_eval/pilot/.


In [3]:
# Mô tả nhanh mẫu pilot so với corpus (chỉ để kiểm tra tính ngẫu nhiên — không dùng nhãn)
by_id = {r['review_id']: r for r in records}
pil = pd.DataFrame([by_id[i] for i in pilot_ids])
cor = pd.DataFrame(records)
print('Phân bố số sao (%) — pilot vs corpus:')
print(pd.DataFrame({'pilot': pil['rating'].value_counts(normalize=True).sort_index() * 100,
                    'corpus': cor['rating'].value_counts(normalize=True).sort_index() * 100}).round(1).to_string())
print('Số từ trung vị — pilot:', int(pil['text_clean'].str.split().str.len().median()),
      '| corpus:', int(cor['text_clean'].str.split().str.len().median()))

Phân bố số sao (%) — pilot vs corpus:
        pilot  corpus
rating               
1        42.8    39.9
2         2.5     3.4
3         3.0     3.2
4         5.2     5.0
5        46.5    48.5


Số từ trung vị — pilot: 7 | corpus: 6


## Đo tỷ lệ nhãn (chạy sau khi hai người gán xong)

Phần này chỉ có kết quả khi đã có `pilot_A.csv` và `pilot_B.csv`. Vì pilot được gán bởi hai người, tôi lấy nhãn đồng thuận; các ca
lệch được xuất ra `pilot_disagreements_to_finalize.csv` để hai người chốt theo guideline v1.2, không tự quyết. Độ đồng thuận trên
pilot cũng là một lần kiểm tra κ bổ sung, lần này trên dữ liệu ngẫu nhiên lớn hơn calibration.

Trước khi chốt các ca lệch, tôi báo tỷ lệ trên phần đồng thuận kèm khoảng chặn dưới/trên (gán mọi ca lệch về phía không thuộc /
thuộc lớp đó). Khi đã có nhãn chốt cho mọi ca lệch, tôi tính tỷ lệ cuối của từng lớp và của nhóm actionable, kèm khoảng tin cậy
95% Wilson — với n = 400 và các lớp hiếm khoảng vài phần trăm, khoảng tin cậy rất quan trọng. Lớp hiếm nhất (p_rare) và khoảng tin
cậy của nó là đầu vào để quyết định kích thước Hold-out ở bước sau.

In [4]:
PA, PB = os.path.join(PILOT_DIR, 'pilot_A.csv'), os.path.join(PILOT_DIR, 'pilot_B.csv')
PILOT_FINALIZE = os.path.join(PILOT_DIR, 'pilot_disagreements_to_finalize.csv')
pilot_df = None
if not (os.path.exists(PA) and os.path.exists(PB)):
    print('Đang chờ pilot_A.csv và pilot_B.csv')
else:
    m = merge(PA, PB, expected_ids=pilot_ids)
    if m['problems']:
        print('Cần sửa tệp gán nhãn trước khi đo:')
        for pr in m['problems']:
            print('   -', pr)
    else:
        pilot_df = m['df']
        r = agreement(pilot_df)
        print(f"Đồng thuận: n={r['n_valid']} | % đồng thuận={r['percent_agreement']} | κ 4 lớp={r['kappa_4class']} "
              f"| κ actionable–OTHER={r['actionable_other']['kappa']} | κ FEATURE–UX={r['feature_ux']['kappa']} "
              f"(n={r['feature_ux']['n_both_feature_or_ux']})")
        print('Ma trận nhầm lẫn (hàng = A, cột = B):')
        print(r['confusion'].to_string())
        groups = disagreements_by_pair(pilot_df)
        print('Số ca lệch:', sum(len(v) for v in groups.values()), '|',
              ', '.join(f'{a}/{b}={len(v)}' for (a, b), v in groups.items()))
        print()
        print('Trước khi chốt ca lệch — tỷ lệ trên phần đồng thuận và khoảng chặn trên toàn pilot:')
        print(pd.DataFrame(prevalence_bounds(pilot_df)).round(4).to_string(index=False))
        if not os.path.exists(PILOT_FINALIZE) and groups:
            write_finalize(build_finalize_rows({'pilot': (PA, PB)}, {}, {}), PILOT_FINALIZE)
            print('Đã xuất ca lệch để chốt nhãn:', PILOT_FINALIZE)

Đang chờ pilot_A.csv và pilot_B.csv


In [5]:
# Tỷ lệ cuối (khi mọi ca lệch đã có agreed_label) + khoảng tin cậy 95% Wilson
if pilot_df is None:
    print('Chưa có dữ liệu gán của pilot.')
else:
    final = {r['review_id']: r['label_a'] for r in pilot_df.to_dict('records') if r['label_a'] == r['label_b']}
    pending = []
    if os.path.exists(PILOT_FINALIZE):
        fin = pd.read_csv(PILOT_FINALIZE, encoding='utf-8-sig', dtype=str).fillna('')
        for r in fin.to_dict('records'):
            lab = r['agreed_label'].strip().upper()
            if lab in ('BUG', 'FEATURE', 'UX', 'OTHER'):
                final[r['review_id']] = lab
            else:
                pending.append(r['review_id'])
    if pending or len(final) != len(pilot_ids):
        print(f'Còn {len(pilot_ids) - len(final)} ca lệch chưa có agreed_label — chưa tính tỷ lệ cuối.')
    else:
        table = prevalence_table(list(final.values()))
        out = pd.DataFrame(table)
        out['tỷ lệ'] = (out['share'] * 100).round(1).astype(str) + '%'
        out['CI 95%'] = out.apply(lambda r: f"{r['ci_low'] * 100:.1f}–{r['ci_high'] * 100:.1f}%", axis=1)
        print(out[['class', 'count', 'tỷ lệ', 'CI 95%']].rename(columns={'class': 'lớp', 'count': 'số ca'}).to_string(index=False))
        rr = rarest(table)
        print(f"\np_rare = {rr['class']}: {rr['share'] * 100:.1f}% (CI 95% {rr['ci_low'] * 100:.1f}–{rr['ci_high'] * 100:.1f}%)")

Chưa có dữ liệu gán của pilot.
